In [1]:
#Task 5
# Türkçe ile İngilizceyi karşılaştır. Aynı paragrafın Türkçesini ve
# İngilizcesini al. GPT-4'ün tokenizer'ıyla (tiktoken, "cl100k_base")
# ikisini de token'la, kelime başına kaç token düştüğünü yaz. Sonra aynı
# Türkçe paragrafı kendi tokenizer'ınla token'la. Türkçenin eklemeli
# yapısı bu sayıları nasıl etkiliyor?

In [2]:
import tiktoken

In [3]:
enc = tiktoken.get_encoding("cl100k_base")

In [4]:
tr = ("Şehirde yaşayanların çoğu, sabahları işe yetişebilmek için saatlerce trafikte bekliyor. "
      "Toplu taşımayı kullanabilecekken arabalarından vazgeçemeyenler, yolları her geçen gün daha da tıkıyor. "
      "Belediyenin açıkladığı yeni planlara göre önümüzdeki yıllarda metro hatları uzatılacak ve otobüslerin sayısı artırılacak. "
      "Yine de uzmanlar, insanların alışkanlıklarını değiştirmedikçe hiçbir yatırımın tek başına sorunu çözemeyeceğini düşünüyor.")

en = ("Most people who live in the city wait in traffic for hours every morning so that they can get to work on time. "
      "Those who could use public transport but cannot give up their cars clog the roads a little more every day. "
      "According to the new plans announced by the municipality, the metro lines will be extended and the number of buses will be increased in the coming years. "
      "Still, experts think that no investment can solve the problem on its own unless people change their habits.")
print(enc.encode(tr))
print("----going for eng now---")
print(enc.encode(en))
print(f' In Turkish, for each word there are {len(enc.encode(tr))/len(tr.split())} GPT-4 tokens')
print(f' In English, for each word there are {len(enc.encode(en))/len(en.split())} GPT-4 tokens')

[129, 252, 2701, 404, 451, 84668, 44957, 96548, 18578, 78, 11257, 84, 11, 19972, 1494, 46937, 55809, 68, 3686, 98902, 3141, 9672, 1247, 34271, 52465, 1565, 346, 99988, 1609, 668, 51033, 747, 48320, 13, 7054, 10036, 9637, 78413, 18864, 3862, 58116, 58209, 46800, 2779, 52412, 8093, 68233, 348, 1394, 713, 3209, 336, 1216, 268, 1565, 11, 379, 21999, 3862, 1077, 77211, 268, 88787, 62594, 3067, 259, 41990, 3862, 48320, 13, 2893, 839, 16618, 268, 258, 64010, 91166, 329, 75437, 99813, 3197, 75, 5169, 31285, 265, 79705, 49621, 79691, 451, 6780, 379, 3862, 657, 72626, 34582, 9072, 46937, 45576, 266, 55888, 79765, 5320, 14479, 677, 2448, 82, 1565, 258, 2019, 90513, 1989, 30333, 55888, 79765, 13, 816, 483, 409, 45576, 1543, 14115, 11, 62638, 96548, 453, 33573, 8826, 26693, 10784, 277, 38511, 409, 44907, 48497, 8600, 1609, 3209, 68, 15960, 3209, 44955, 379, 266, 30333, 38404, 16507, 43185, 47655, 64566, 25551, 63353, 18578, 3029, 95233, 19494, 346, 11257, 6729, 52119, 7370, 16461, 2448, 48320, 13]


In [5]:
with open("altayli_kose_yazilari.txt", "r", encoding = "utf-8") as f:
    text = f.read()

tokens = text.encode("utf-8")
tokens = list(map(int,tokens))

In [6]:
def get_stats(ids):
    counts = {}
    for pair in zip(ids,ids[1:]):
        counts[pair] = counts.get(pair,0)+1
    return counts

In [7]:
def merge (ids, pair, idx):
    newids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair [1]:
            newids.append(idx)
            i += 2 
        else:
            newids.append(ids [i])
            i += 1
    return newids

In [8]:
merges = {} 
ids = list(tokens)
for i in range(50):
    stats = get_stats(ids)
    pair = max(stats, key=stats.get)
    idx = 256 + i
    ids = merge(ids, pair, idx)
    merges[pair] = idx

In [9]:
vocab = {idx: bytes([idx]) for idx in range(256)}
for (p0,p1), idx in merges.items():
    vocab[idx] = vocab[p0] + vocab[p1]
    
#encode string to integers
def decode(ids):
    tokens = b''.join(vocab[idx] for idx in ids)
    text = tokens.decode('utf-8', errors='replace')
    return text

def encode(text):
    tokens = list(text.encode('utf-8'))
    while len(tokens) >= 2:
        stats=get_stats(tokens)
        pair = min(stats, key= lambda p: merges.get(p, float("inf")))
        if pair not in merges:
            break
        idx = merges[pair]
        tokens = merge(tokens, pair, idx)
    return tokens

In [10]:
print(encode(tr))
print("----going for eng now---")
print(encode(en))

print(f' In Turkish, for each word there are {len(encode(tr))/len(tr.split())} Homemade tokens :D')
print(f' In English, for each word there are {len(encode(en))/len(en.split())} Homemade tokens ')

[197, 158, 101, 104, 273, 299, 283, 262, 97, 121, 269, 278, 304, 271, 111, 268, 117, 270, 115, 97, 302, 104, 278, 272, 105, 262, 258, 121, 288, 105, 262, 101, 98, 274, 303, 275, 105, 271, 293, 115, 97, 97, 116, 108, 264, 99, 258, 116, 114, 97, 102, 105, 107, 116, 258, 98, 101, 107, 108, 286, 277, 46, 32, 84, 111, 112, 108, 287, 116, 97, 262, 256, 280, 121, 272, 107, 117, 108, 108, 269, 97, 98, 274, 101, 99, 101, 107, 107, 101, 257, 259, 97, 302, 278, 282, 294, 257, 118, 97, 122, 103, 101, 271, 101, 303, 121, 292, 108, 264, 270, 121, 300, 278, 272, 104, 264, 32, 103, 101, 271, 101, 257, 103, 261, 257, 294, 104, 260, 298, 116, 256, 107, 256, 121, 277, 46, 32, 66, 291, 101, 100, 286, 292, 293, 97, 271, 256, 107, 285, 301, 268, 272, 121, 292, 265, 112, 108, 269, 278, 260, 103, 279, 114, 258, 279, 110, 261, 109, 261, 122, 281, 107, 265, 121, 256, 108, 278, 298, 109, 288, 114, 111, 32, 104, 97, 116, 278, 272, 117, 122, 97, 116, 256, 285, 99, 97, 275, 118, 258, 111, 116, 111, 98, 261, 115, 10

In [11]:
print(encode("kullanabilecekken"))
print(enc.encode("kullanabilecekken"))

[107, 117, 108, 108, 269, 97, 98, 274, 101, 99, 101, 107, 107, 292]
[74, 42220, 58209, 46800, 2779]


In [12]:
print([enc.decode([t]) for t in enc.encode(" kullanabilecekken")])
print([decode([t]) for t in encode("kullanabilecekken")])

[' kullan', 'abile', 'cek', 'ken']
['k', 'u', 'l', 'l', 'an', 'a', 'b', 'il', 'e', 'c', 'e', 'k', 'k', 'en']
